In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent   # nbconvert and Jupyter both start the kernel in notebooks/
assert (ROOT / "paths.py").is_file(), f"paths.py not found from {Path.cwd()}"
sys.path.insert(0, str(ROOT))

# Step 5, second learner: FLANN-style ridge regression

Same target, folds, fold baselines, training rows and benchmark as `residual_lobo.ipynb`. The learner is ridge regression on a fixed functional-link basis (`residual.make_flann_basis`, 39 columns), set in `docs/plan.md` before the run:

- daily harmonics 1-3 of LTST and seasonal harmonics 1-2 of Ls, each times (1, tau), and tau;
- linear TI, slope, sin/cos aspect, albedo and the baseline, each times (1, sin, cos of the first daily harmonic);
- no elevation; standardized columns; ridge alpha 1.0, fixed.

**Decision rule (fixed before running):** adopt FLANN over `no_elev` (gradient boosting) only if its macro RMSE over the 9 blocks is at least 0.2 K lower **and** it is better in at least 5 of 9 blocks. This is configuration 5.

Fold 0 is reported on its own: block 0's TI (566-682 tiu) lies above every other block's (at most 514 tiu), and FLANN is the only learner here that extrapolates in TI (linearly).

In [ ]:
# --- config ---
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import paths
import thermal1d as t
import residual as r

SUSPECT_SOLS = (3763, 3778)
EXPECTED_TRAIN_BLOCKS = [0, 1, 2, 3, 5, 6, 7, 9, 10]
LS_BIN_DEG = 30
ADOPT_MIN_GAIN_K, ADOPT_MIN_BLOCK_WINS = 0.2, 5
REPRO_TOL_K = 0.05
NIGHT_HOURS = [0, 1, 2, 3, 4, 5, 19, 20, 21, 22, 23]
TI_CURVE = np.linspace(300.0, 750.0, 46)   # tiu, for the TI response curve
FIT_PARAMS = paths.OUT_DIR / "fit_params.csv"
STEP5_ROWS = paths.RESIDUAL_OUT_DIR / "residual_rows.csv"
OUT = paths.RESIDUAL_OUT_DIR
print("FLANN alpha:", r.FLANN_RIDGE_ALPHA, "| daily harmonics:", r.FLANN_DAILY_HARMONICS,
      "| seasonal harmonics:", r.FLANN_SEASON_HARMONICS)

## Data and fold baselines (as in `residual_lobo.ipynb`)

In [ ]:
tbl = pd.DataFrame(t.load_rems_table(paths.DATA_DIR / "gale_rems_training.csv"))
assert (tbl.split_role == "train").all()
blocks = sorted(tbl.block.unique())
assert blocks == EXPECTED_TRAIN_BLOCKS, f"training blocks {blocks}"
per_sol_cols = ["ls_deg", "tau", "tau_is_placeholder", "row", "slope_deg", "aspect_deg",
                "albedo_mean", "ti_themis_tiu", "block"]
assert (tbl.groupby("sol")[per_sol_cols].nunique() == 1).all().all(), "per-sol inputs vary within a sol"
sols = tbl.groupby("sol")[per_sol_cols].first().reset_index()
sol_inputs = {"lat_deg": t.lat_from_row(sols.row.to_numpy()), "ls_deg": sols.ls_deg.to_numpy(),
              "tau": sols.tau.to_numpy(), "slope_deg": sols.slope_deg.to_numpy(),
              "aspect_deg": sols.aspect_deg.to_numpy(), "albedo": sols.albedo_mean.to_numpy(),
              "ti": sols.ti_themis_tiu.to_numpy()}
row_sol = pd.Series(np.arange(len(sols)), index=sols.sol)[tbl.sol].to_numpy()
row_step = t.bin_centre_step(tbl.bin.to_numpy())
evaluate = t.make_evaluator(sol_inputs, row_sol, row_step)

obs = tbl.ground_temp_k.to_numpy()
block = tbl.block.to_numpy()
suspect = tbl.sol.between(*SUSPECT_SOLS).to_numpy()
placeholder = tbl.tau_is_placeholder.to_numpy()
fit_ok = ~suspect & ~placeholder
tbl["group"] = np.where(suspect, "suspect_terrain", np.where(placeholder, "placeholder_tau", "main"))
tbl["ltst_hour"] = (tbl.bin // 4).astype(int)
tbl["ltst_hr"] = (tbl.bin + 0.5) * t.BIN_HOURS
tbl["ls_bin"] = (tbl.ls_deg // LS_BIN_DEG * LS_BIN_DEG).astype(int)
tbl["q_abs_wm2"] = 0.0     # not in the FLANN basis; make_features needs the column

step5 = pd.read_csv(STEP5_ROWS)
assert (step5.sol.to_numpy() == tbl.sol.to_numpy()).all() and (step5.bin.to_numpy() == tbl.bin.to_numpy()).all(), \
    "row order differs from Step 5"
fits = pd.read_csv(FIT_PARAMS).set_index("fold")
fold_base = {}
t0 = time.time()
for label in fits.index:
    p = tuple(fits.loc[label, list(t.PARAM_NAMES)].to_numpy(dtype=float))
    fold_base[label] = evaluate(p).copy()
    assert np.all(np.isfinite(fold_base[label])), f"non-finite baseline in {label}"
print(f"fold baselines in {time.time() - t0:.0f} s")
oof = np.full(len(tbl), np.nan)
for b in blocks:
    oof[block == b] = fold_base[f"block {b}"][block == b]
diff = np.abs(oof - step5.baseline_k.to_numpy()).max()
print(f"max |fold baseline - Step 5 baseline_k| = {diff:.4f} K")
assert diff < REPRO_TOL_K, "fold baselines do not reproduce Step 5"
tbl["baseline_k"] = oof
tbl["bench_k"] = step5.bench_k.to_numpy()
tbl["pred_no_elev"] = step5.pred_no_elev.to_numpy()

## LOBO with FLANN

In [ ]:
tbl["pred_flann"] = np.nan
models = {}
for b in blocks + ["all"]:
    label = f"block {b}" if b != "all" else "all training blocks"
    held = block == b if b != "all" else np.zeros(len(tbl), bool)
    train = fit_ok & ~held
    B = r.make_flann_basis(tbl.assign(baseline_k=fold_base[label]))
    model = r.make_flann_learner().fit(B[train], obs[train] - fold_base[label][train])
    models[b] = model
    if held.any():
        tbl.loc[held, "pred_flann"] = fold_base[label][held] + model.predict(B[held])
assert tbl.pred_flann.notna().all()
n_basis = B.shape[1]
print("basis columns:", n_basis)

## Scores and the decision rule

In [ ]:
MODELS = ["baseline", "bench", "no_elev", "flann"]
COLS = {"baseline": "baseline_k", "bench": "bench_k", "no_elev": "pred_no_elev", "flann": "pred_flann"}
for m in MODELS:
    tbl[f"resid_{m}"] = tbl[COLS[m]] - tbl.ground_temp_k


def score(g, models=MODELS):
    out = {"n": len(g)}
    for m in models:
        out[f"{m}_rmse_k"] = np.sqrt((g[f"resid_{m}"] ** 2).mean())
        out[f"{m}_bias_k"] = g[f"resid_{m}"].mean()
    return pd.Series(out)


scored = tbl[tbl.group != "placeholder_tau"]
main = tbl[tbl.group == "main"]
by_block = scored.groupby("block").apply(score, include_groups=False)
by_group = tbl.groupby("group").apply(score, include_groups=False)
summary = pd.DataFrame({
    "macro_rmse_k": {m: by_block[f"{m}_rmse_k"].mean() for m in MODELS},
    "macro_rmse_no_3763_k": {m: by_block.drop(index=10)[f"{m}_rmse_k"].mean() for m in MODELS},
    "macro_rmse_no_block0_k": {m: by_block.drop(index=0)[f"{m}_rmse_k"].mean() for m in MODELS},
    "pooled_rmse_main_k": {m: np.sqrt((main[f"resid_{m}"] ** 2).mean()) for m in MODELS},
    "rmse_placeholder_tau_k": {m: by_group.loc["placeholder_tau", f"{m}_rmse_k"] for m in MODELS},
    "rmse_suspect_terrain_k": {m: by_group.loc["suspect_terrain", f"{m}_rmse_k"] for m in MODELS},
    "blocks_beating_bench": {m: int((by_block[f"{m}_rmse_k"] < by_block.bench_rmse_k).sum()) for m in MODELS},
})
pd.options.display.float_format = "{:.2f}".format
print(summary.to_string())
print()
print(by_block.to_string())

gain = summary.macro_rmse_k["no_elev"] - summary.macro_rmse_k["flann"]
wins = int((by_block.flann_rmse_k < by_block.no_elev_rmse_k).sum())
adopt = gain >= ADOPT_MIN_GAIN_K and wins >= ADOPT_MIN_BLOCK_WINS
decision = (f"FLANN vs no_elev: macro gain {gain:+.3f} K (need >= {ADOPT_MIN_GAIN_K}), better in {wins}/9 blocks "
            f"(need >= {ADOPT_MIN_BLOCK_WINS}) -> {'ADOPT flann' if adopt else 'keep no_elev'}")
print()
print(decision)
(OUT / "flann_decision.txt").write_text(decision + chr(10))

In [ ]:
b0 = tbl[tbl.block == 0]
night0 = b0[b0.ltst_hour.isin(NIGHT_HOURS)]
day0 = b0[~b0.ltst_hour.isin(NIGHT_HOURS)]
fold0 = pd.DataFrame({m: {"rmse_k": np.sqrt((b0[f"resid_{m}"] ** 2).mean()), "bias_k": b0[f"resid_{m}"].mean(),
                          "night_bias_k": night0[f"resid_{m}"].mean(), "day_bias_k": day0[f"resid_{m}"].mean()}
                      for m in MODELS}).T
print("Fold 0 (block 0 held out; its TI is outside the training range):")
print(fold0.to_string())

by_hour = main.groupby("ltst_hour").apply(score, include_groups=False)
by_ls = main.groupby("ls_bin").apply(score, include_groups=False)
print()
print("By LTST hour (main rows):")
print(by_hour[[f"{m}_bias_k" for m in MODELS] + [f"{m}_rmse_k" for m in MODELS]].to_string())
print()
print("By Ls bin (main rows):")
print(by_ls[["n"] + [f"{m}_bias_k" for m in MODELS]].to_string())

## TI response and map safety

Mean predicted residual as TI is swept over 300-750 tiu with every other input at the training rows' values. For the fold-0 model (trained without block 0, TI at most 514 tiu) and the all-blocks model. Beyond the training range a linear model extrapolates by assumption.

In [ ]:
def ti_curve(model, label, rows):
    frame = tbl[rows].assign(baseline_k=fold_base[label][rows])
    return np.array([model.predict(r.make_flann_basis(frame.assign(ti_themis_tiu=v))).mean() for v in TI_CURVE])


curves = pd.DataFrame({"ti_tiu": TI_CURVE,
                       "fold0_model": ti_curve(models[0], "block 0", fit_ok & (block != 0)),
                       "all_blocks_model": ti_curve(models["all"], "all training blocks", fit_ok)})
curves.to_csv(OUT / "flann_ti_response.csv", index=False)
train_max0 = tbl.ti_themis_tiu[fit_ok & (block != 0)].max()
print(f"fold-0 training TI max: {train_max0:.0f} tiu")
print(curves.iloc[::5].to_string(index=False))

coef = pd.Series(models["all"][-1].coef_, index=r.make_flann_basis(tbl.head(1)).columns)
print()
print("All-blocks model, standardized coefficients of the static terms (K per standard deviation):")
print(coef[[c for c in coef.index if any(c.startswith(s) for s in r.FLANN_STATIC)]].to_string())

## Covariate effects by hour, inside and beyond the rover's range

For each static covariate, the mean predicted residual per LTST hour as the covariate is swept (other inputs at the training rows). Effect = the largest spread over the sweep at any hour. "Rover range" is the training rows' 1st-99th percentile; "extended" is a range the map will reach. Linear terms extrapolate, so the extended effect grows with the range.

In [ ]:
def hourly_effect(model, label, rows, col, values):
    frame = tbl[rows].assign(baseline_k=fold_base[label][rows])
    hour = frame.ltst_hour.to_numpy()
    out = []
    for v in values:
        pred = model.predict(r.make_flann_basis(frame.assign(**{col: v})))
        out.append(pd.Series(pred).groupby(hour).mean())
    return pd.DataFrame(out, index=values)


EXTENDED = {"ti_themis_tiu": (300.0, 750.0), "slope_deg": (0.0, 30.0), "albedo_mean": (0.10, 0.35)}
rows_all = fit_ok
eff = []
for col, (elo, ehi) in EXTENDED.items():
    lo, hi = np.percentile(tbl.loc[rows_all, col], [1, 99])
    for rng_name, (a, b) in (("rover range", (lo, hi)), ("extended", (elo, ehi))):
        h = hourly_effect(models["all"], "all training blocks", rows_all, col, np.linspace(a, b, 11))
        spread = h.max() - h.min()
        eff.append({"covariate": col, "range": rng_name, "lo": a, "hi": b,
                    "max_spread_k": spread.max(), "at_hour": int(spread.idxmax())})
eff = pd.DataFrame(eff)
eff.to_csv(OUT / "flann_covariate_effects.csv", index=False)
print("All-blocks FLANN model, largest hourly spread of the predicted residual:")
print(eff.to_string(index=False))

h0 = hourly_effect(models[0], "block 0", fit_ok & (block != 0), "ti_themis_tiu", np.array([450.0, 514.0, 620.0]))
night_cols = [c for c in h0.columns if c in NIGHT_HOURS]
print()
print("Fold-0 model (trained without block 0): mean predicted residual at TI 450 / 514 / 620 tiu")
print("night (19-06 h):", np.round(h0[night_cols].mean(axis=1).to_numpy(), 2), "K")
print("day (07-18 h):  ", np.round(h0[[c for c in h0.columns if c not in NIGHT_HOURS]].mean(axis=1).to_numpy(), 2), "K")

In [ ]:
tbl[["sol", "bin", "block", "group", "ground_temp_k", "baseline_k", "bench_k", "pred_no_elev", "pred_flann"]] \
    .to_csv(OUT / "flann_rows.csv", index=False)
summary.to_csv(OUT / "flann_summary.csv")
by_block.to_csv(OUT / "flann_by_block.csv")
by_hour.to_csv(OUT / "flann_by_ltst_hour.csv")
by_ls.to_csv(OUT / "flann_by_ls_bin.csv")
fold0.to_csv(OUT / "flann_fold0.csv")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ax = axes[0]
x = np.arange(len(by_block))
for i, m in enumerate(["bench", "no_elev", "flann"]):
    ax.bar(x + (i - 1) * 0.27, by_block[f"{m}_rmse_k"], 0.27, label=m)
ax.set_xticks(x, [str(b) for b in by_block.index])
ax.set(xlabel="held-out block", ylabel="RMSE (K)", title="Leave-one-block-out RMSE")
ax.legend(fontsize=8)
ax = axes[1]
for m, st in (("baseline", "r-"), ("bench", "b--"), ("no_elev", "g-"), ("flann", "m-")):
    ax.plot(by_hour.index, by_hour[f"{m}_bias_k"], st, label=m)
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="LTST hour", ylabel="bias (K)", title="Bias by LTST hour (main rows)")
ax.legend(fontsize=8)
ax = axes[2]
ax.plot(curves.ti_tiu, curves.fold0_model, "m-", label="fold-0 model (no block 0)")
ax.plot(curves.ti_tiu, curves.all_blocks_model, "k-", label="all-blocks model")
ax.axvline(train_max0, color="m", ls=":", label="fold-0 training TI max")
ax.axvspan(tbl.ti_themis_tiu[block == 0].min(), tbl.ti_themis_tiu[block == 0].max(), color="grey", alpha=0.2,
           label="block 0 TI range")
ax.set(xlabel="TI (tiu)", ylabel="mean predicted residual (K)", title="FLANN TI response")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUT / "flann_scores.png", dpi=120)
print("saved to", OUT)